In [0]:
%sql
-- ============================================================
-- ETL: silver.ventas + silver.categorias → gold.dim_producto
-- Patrón: SCD Type 2 sobre titulo_producto
-- 
-- PASO 1 — MERGE: cierra filas activas cuyo título cambió
-- PASO 2 — INSERT: agrega filas nuevas (productos nuevos o
--          títulos que cambiaron)
-- ============================================================


-- =====================================================
-- FUENTE: producto actual por id_producto
-- Título más reciente desde Silver + atributos de categoría
-- Esta CTE se usa en ambos pasos
-- =====================================================
WITH productos_actuales AS (
    SELECT
        v.id_producto,
        v.titulo_producto,
        v.categoria_id,
        c.nivel_1,
        c.nivel_2,
        c.nombre                                    AS nombre_categoria,
        CAST(MIN(v.fecha) AS DATE)                  AS primera_venta,
        ROW_NUMBER() OVER (
            PARTITION BY v.id_producto
            ORDER BY MAX(v.fecha) DESC
        )                                           AS rn
    FROM iniciacion_deportiva.silver.ventas v
    LEFT JOIN iniciacion_deportiva.silver.categorias c
        ON v.categoria_id = c.categoria_id
    GROUP BY
        v.id_producto,
        v.titulo_producto,
        v.categoria_id,
        c.nivel_1,
        c.nivel_2,
        c.nombre
),

fuente AS (
    SELECT
        id_producto,
        titulo_producto,
        categoria_id,
        nivel_1,
        nivel_2,
        nombre_categoria,
        primera_venta
    FROM productos_actuales
    WHERE rn = 1    -- título más reciente por producto
)

-- =====================================================
-- PASO 1: Cerrar filas activas cuyo título cambió
-- Cuando el título en Silver difiere del título activo
-- en Gold → cerramos la fila (fecha_fin = hoy, es_actual = FALSE)
-- =====================================================
MERGE INTO iniciacion_deportiva.gold.dim_producto AS target
USING fuente AS source
ON target.id_producto = source.id_producto
   AND target.es_actual = TRUE

WHEN MATCHED AND target.titulo_producto != source.titulo_producto
    THEN UPDATE SET
        target.fecha_fin   = CURRENT_DATE() - INTERVAL 1 DAY,
        target.es_actual   = FALSE,
        target._updated_at = CURRENT_TIMESTAMP();


-- =====================================================
-- PASO 2: Insertar filas nuevas
-- Cubre dos casos:
--   a) Producto nuevo — no existe en Gold
--   b) Título cambió — la fila vieja ya fue cerrada en PASO 1,
--      ahora insertamos la fila nueva con el título actualizado
-- =====================================================
INSERT INTO iniciacion_deportiva.gold.dim_producto (
    id_producto,
    titulo_producto,
    categoria_id,
    nivel_1,
    nivel_2,
    nombre_categoria,
    fecha_inicio,
    fecha_fin,
    es_actual,
    _source,
    _created_at,
    _updated_at
)

WITH productos_actuales AS (
    SELECT
        v.id_producto,
        v.titulo_producto,
        v.categoria_id,
        c.nivel_1,
        c.nivel_2,
        c.nombre                                    AS nombre_categoria,
        CAST(MIN(v.fecha) AS DATE)                  AS primera_venta,
        ROW_NUMBER() OVER (
            PARTITION BY v.id_producto
            ORDER BY MAX(v.fecha) DESC
        )                                           AS rn
    FROM iniciacion_deportiva.silver.ventas v
    LEFT JOIN iniciacion_deportiva.silver.categorias c
        ON v.categoria_id = c.categoria_id
    GROUP BY
        v.id_producto,
        v.titulo_producto,
        v.categoria_id,
        c.nivel_1,
        c.nivel_2,
        c.nombre
),

fuente AS (
    SELECT
        id_producto,
        titulo_producto,
        categoria_id,
        nivel_1,
        nivel_2,
        nombre_categoria,
        primera_venta
    FROM productos_actuales
    WHERE rn = 1
)

SELECT
    f.id_producto,
    f.titulo_producto,
    f.categoria_id,
    f.nivel_1,
    f.nivel_2,
    f.nombre_categoria,
    -- fecha_inicio: primera venta del producto si es nuevo,
    -- hoy si es un título que cambió
    COALESCE(
        CASE WHEN d.producto_key IS NULL THEN f.primera_venta END,
        CURRENT_DATE()
    )                       AS fecha_inicio,
    DATE '9999-12-31'       AS fecha_fin,
    TRUE                                            AS es_actual,
    'iniciacion_deportiva.silver.ventas'            AS _source,
    CURRENT_TIMESTAMP()                             AS _created_at,
    CURRENT_TIMESTAMP()                             AS _updated_at
FROM fuente f
-- Solo insertamos si NO existe ya una fila activa con ese título
LEFT JOIN iniciacion_deportiva.gold.dim_producto d
    ON f.id_producto = d.id_producto
    AND f.titulo_producto = d.titulo_producto
    AND d.es_actual = TRUE
WHERE d.producto_key IS NULL;